# 옷장패션 데이터 정제 보고서

## 1. 데이터 개요
- 행/열: 1,500 × 8
- 주요 컬럼: order_id, customer_age, category, channel, price, quantity, amount, return_amount

## 2. 진단 결과
- 결측: amount 3.40%(51건), price 0.33%(5건), 나머지 열은 결측 없음
- 숨은 결측: customer_age의 0, 999는 실제 나이가 아니라 '모름'·입력 실수로 추정
- 이상치(IQR): customer_age(999·0, IQR 18건 중 실제 오류 5건), quantity(200, 1건), amount(50,000,000, IQR 145건 중 실제 오류 1건)
  - IQR은 정상 값도 잡아 '후보'로만 활용: customer_age는 IQR 범위(12.6~53.6세) 밖의 정상 연령 13건 포함, amount는 고가 상품 2개 주문(259,800원)도 상한(253,200원)을 넘음
  - return_amount는 25~75%가 모두 0이라 IQR 적용 부적합 (122건 = 반품 발생 건수)
  - price는 IQR 이상치 0건, 15,900~129,900원으로 정상 범위
- 의심되는 결측 유형: amount 결측의 88%가 app 채널(전체 app 비율 61%) → 채널과 관련된 결측, MAR

## 3. 처리 결정과 근거
| 이슈 | 결정 | 근거 | 한계 |
| --- | --- | --- | --- |
| amount 결측 | 채널별 중앙값 대체 | app 결측 비율↑ → MAR 가설에 부합 | 채널 외 다른 원인은 검토 안 함 |
| price 결측 | 카테고리별 중앙값 대체 | 비율 0.3%로 적음, 카테고리별 가격대 보존 | 표본 적어 카테고리 통계 불안정 |
| customer_age = 999, 0 | NaN 표시 → 중앙값 대체 | 물리적 불가능 — 입력 실수 추정 | 외부 인증 데이터 없음 |
| quantity = 200 | NaN 표시 → 중앙값 대체 | 일반 소비자 도메인 외 값 | 만약 도매 가능 고객이면 제거가 손실 |
| amount = 50,000,000 | 유지 + amount_outlier 플래그 | 1건의 비즈니스적 가능성 보존 | 도매면 별도 분석 필요 |

- 처리 기준: IQR 대신 도메인 기준 사용 (customer_age 1 미만·110 초과, quantity 100 이상, amount 5천만 원 이상)
- 오류 값은 먼저 NaN으로 바꾼 뒤 중앙값을 계산하여 오류 값이 대체값에 섞이지 않게 함
- 원본(partner)은 보존하고 복사본(partner_clean)에서 처리

## 4. 처리 후 검증
- 결측 0건 (price 5 → 0, amount 51 → 0, 설계상 NaN 유지가 필요한 컬럼 없음)
- customer_age 범위: 5 ~ 60 (0, 999 총 5건 제거 확인)
- quantity 최댓값: 3 (200 제거 확인)
- amount_outlier 플래그 1건 보존

## 5. 후속 권고
- 도매 가능 고객 식별을 위해 customer_id 단위 과거 구매 이력 확보 필요
- amount 50,000,000은 price 최댓값(129,900) × quantity 최댓값(200) = 약 2,600만 원보다 커 계산상 불가능 → 원천 데이터에서 실제 금액 확인 필요
- amount 결측은 price × quantity로 재계산하면 중앙값보다 정확하게 복원 가능 → 다음 정제 시 적용 검토
- app 채널의 amount 결측 원인(기록 누락·버그 등)을 개발팀에 확인 요청
- 평균 등 통계 산출 시 amount_outlier = 1 행은 제외하고 계산
- customer_age 최솟값 5세는 직접 구매 주체로 보기 어려움 → 보호자 계정 여부 또는 입력 오류 확인 필요 (현재 기준 1세 미만만 처리)